## NPPES: address-only extracts (ZIP / street / city / state)

Per **NPPES Data Dissemination Readme v.2** (bundle PDF), the CSV bundle includes:

- **`npidata_pfile_*.csv`** — one row per NPI. FOIA-disclosable **business mailing** and **primary practice location** address fields (street lines, city, state, postal code, country). Telephone/fax are excluded here so the extract is **address-focused**.
- **`pl_pfile_*.csv`** — **secondary (non-primary) practice locations** for Type 1 and Type 2 NPIs (again street, city, state, postal, country; no phone/fax).
- **`endpoint_pfile_*.csv`** — endpoints plus optional **affiliation mailing-style** fields (`Affiliation Address …`), included below as another address table keyed by NPI.

**`NPPES_Data_Dissemination_CodeValues.pdf`** documents coded values (entity type, endpoint types, etc.), not street addresses; it is useful for interpreting other columns if you widen the extract later.

Postal codes in the dissemination file are stored as **VARCHAR** (often 9 digits without a hyphen for US ZIP+4).

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

REPO = Path("..").resolve()
NPPES = REPO / "NPPES_Data_Dissemination_March_2026_V2"
OUT_DIR = REPO / "data" / "nppes_addresses"
OUT_DIR.mkdir(parents=True, exist_ok=True)

NPIDATA_CSV = NPPES / "npidata_pfile_20050523-20260308.csv"
NPIDATA_HDR = NPPES / "npidata_pfile_20050523-20260308_fileheader.csv"
PL_CSV = NPPES / "pl_pfile_20050523-20260308.csv"
PL_HDR = NPPES / "pl_pfile_20050523-20260308_fileheader.csv"
ENDPOINT_CSV = NPPES / "endpoint_pfile_20050523-20260308.csv"
ENDPOINT_HDR = NPPES / "endpoint_pfile_20050523-20260308_fileheader.csv"

NPPES.exists(), NPIDATA_CSV.exists()

(True, True)

In [2]:
def _read_header_columns(header_csv: Path) -> list[str]:
    return list(pd.read_csv(header_csv, nrows=0).columns)


def npidata_address_columns(columns: list[str]) -> list[str]:
    """Mailing + primary practice location address fields; drop phone/fax."""
    out: list[str] = []
    for c in columns:
        if c == "NPI":
            continue
        if "Telephone" in c or "Fax" in c or "Extension" in c:
            continue
        if "Business Mailing Address" in c or "Business Practice Location Address" in c:
            out.append(c)
    return ["NPI"] + out


def pl_address_columns(columns: list[str]) -> list[str]:
    """Secondary practice location file: keep street/city/state/ZIP/country only."""
    out: list[str] = []
    for c in columns:
        if c == "NPI":
            continue
        if any(x in c for x in ["Telephone", "Fax", "Extension"]):
            continue
        if "Secondary Practice Location" in c or "Practice Location Address" in c:
            out.append(c)
    return ["NPI"] + out


def endpoint_affiliation_address_columns(columns: list[str]) -> list[str]:
    return ["NPI"] + [c for c in columns if c != "NPI" and "Affiliation Address" in c]


npidata_cols = npidata_address_columns(_read_header_columns(NPIDATA_HDR))
pl_cols = pl_address_columns(_read_header_columns(PL_HDR))
endpoint_cols = endpoint_affiliation_address_columns(_read_header_columns(ENDPOINT_HDR))

len(npidata_cols), len(pl_cols), len(endpoint_cols)

(13, 7, 7)

In [3]:
pd.DataFrame(
    {
        "file": ["npidata", "pl_pfile", "endpoint"],
        "n_columns": [len(npidata_cols), len(pl_cols), len(endpoint_cols)],
    }
)

,file,n_columns
0,npidata,13
1,pl_pfile,7
2,endpoint,7


### Preview (first few rows)

Set `PREVIEW_NROWS` to control how many rows to load into memory for each file.

In [4]:
PREVIEW_NROWS = 5

df_npi_preview = pd.read_csv(
    NPIDATA_CSV,
    nrows=PREVIEW_NROWS,
    usecols=npidata_cols,
    dtype=str,
    low_memory=False,
)
df_pl_preview = pd.read_csv(
    PL_CSV,
    nrows=PREVIEW_NROWS,
    usecols=pl_cols,
    dtype=str,
    low_memory=False,
)
df_ep_preview = pd.read_csv(
    ENDPOINT_CSV,
    nrows=PREVIEW_NROWS,
    usecols=endpoint_cols,
    dtype=str,
    low_memory=False,
)

df_npi_preview

,NPI,Provider First Line Business Mailing Address,Provider Second Line Business Mailing Address,Provider Business Mailing Address City Name,Provider Business Mailing Address State Name,Provider Business Mailing Address Postal Code,Provider Business Mailing Address Country Code (If outside U.S.),Provider First Line Business Practice Location Address,Provider Second Line Business Practice Location Address,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,Provider Business Practice Location Address Postal Code,Provider Business Practice Location Address Country Code (If outside U.S.)
0,1679576722,PO BOX 2168,NaN,KEARNEY,NE,688482168,US,3500 CENTRAL AVE,NaN,KEARNEY,NE,688472944,US
1,1588667638,1824 KING STREET,SUITE 300,JACKSONVILLE,FL,322044736,US,1824 KING STREET,SUITE 300,JACKSONVILLE,FL,322044736,US
2,1497758544,3418 VILLAGE DR,NaN,FAYETTEVILLE,NC,283044552,US,3418 VILLAGE DR,NaN,FAYETTEVILLE,NC,283044552,US
3,1306849450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1215930367,17323 RED OAK DR,NaN,HOUSTON,TX,770901243,US,17323 RED OAK DR,NaN,HOUSTON,TX,770901243,US


In [5]:
df_pl_preview

,NPI,Provider Secondary Practice Location Address- Address Line 1,Provider Secondary Practice Location Address- Address Line 2,Provider Secondary Practice Location Address - City Name,Provider Secondary Practice Location Address - State Name,Provider Secondary Practice Location Address - Postal Code,Provider Secondary Practice Location Address - Country Code (If outside U.S.)
0,1154324382,7800 Sheridan St,NaN,Pembroke Pines,FL,330242536,US
1,1154324382,500 N Hiatus Rd Ste 200,NaN,Pembroke Pines,FL,330265213,US
2,1699778829,2200 Cedarcrest Dr,Ste A,Rice Lake,WI,54868,US
3,1699778829,757 Lakeland Dr Ste B,NaN,Chippewa Falls,WI,547291672,US
4,1124021324,31 S Stanfield Rd Ste 104,NaN,Troy,OH,453732334,US


In [6]:
df_ep_preview

,NPI,Affiliation Address Line One,Affiliation Address Line Two,Affiliation Address City,Affiliation Address State,Affiliation Address Country,Affiliation Address Postal Code
0,1154324382,3501 Johnson St,NaN,Hollywood,FL,US,330215421
1,1154324382,500 N Hiatus,Ste 200,Pembroke Pines,FL,US,33026
2,1154324366,1605 S 31st St,NaN,Temple,TX,US,765089299
3,1962405175,1501 N Cedar Crest Blvd,suite 110,Allentown,PA,US,181042309
4,1699778894,2401 W University Ave,NaN,Muncie,IN,US,47303


### Export full address-only CSVs

The NPI data file is very large; it is streamed in **chunks**. Outputs go to `data/nppes_addresses/` (ignored by `.gitignore` if `data/` is listed).

- Set `MAX_NPI_ROWS = None` to write the **entire** NPI address extract (long run, large disk).
- Set `MAX_NPI_ROWS` to an integer for a bounded export while testing.

In [7]:
CHUNK_ROWS = 100_000
MAX_NPI_ROWS = None  # e.g. 50_000 for a smaller test export

out_npi = OUT_DIR / "npidata_addresses_only.csv"
out_pl = OUT_DIR / "pl_secondary_practice_addresses_only.csv"
out_ep = OUT_DIR / "endpoint_affiliation_addresses_only.csv"


def export_npidata_addresses_chunked(
    src: Path,
    dst: Path,
    usecols: list[str],
    *,
    chunksize: int,
    max_rows: int | None,
) -> int:
    first = True
    written = 0
    reader = pd.read_csv(
        src,
        usecols=usecols,
        chunksize=chunksize,
        dtype=str,
        low_memory=False,
    )
    for chunk in reader:
        if max_rows is not None:
            remain = max_rows - written
            if remain <= 0:
                break
            if len(chunk) > remain:
                chunk = chunk.iloc[:remain].copy()
        chunk.to_csv(dst, mode="w" if first else "a", index=False, header=first)
        first = False
        written += len(chunk)
        if max_rows is not None and written >= max_rows:
            break
    return written


n_npi = export_npidata_addresses_chunked(
    NPIDATA_CSV,
    out_npi,
    npidata_cols,
    chunksize=CHUNK_ROWS,
    max_rows=MAX_NPI_ROWS,
)

df_pl = pd.read_csv(PL_CSV, usecols=pl_cols, dtype=str, low_memory=False)
df_pl.to_csv(out_pl, index=False)

df_ep = pd.read_csv(ENDPOINT_CSV, usecols=endpoint_cols, dtype=str, low_memory=False)
df_ep.to_csv(out_ep, index=False)

{
    "npidata_address_rows_written": n_npi,
    "pl_rows": len(df_pl),
    "endpoint_rows": len(df_ep),
    "outputs": [str(out_npi), str(out_pl), str(out_ep)],
}

{'npidata_address_rows_written': 9415362,
 'pl_rows': 1160011,
 'endpoint_rows': 594515,
 'outputs': ['/Users/shashin/Documents/GitHub/INT Biohackathon 2026/data/nppes_addresses/npidata_addresses_only.csv',
  '/Users/shashin/Documents/GitHub/INT Biohackathon 2026/data/nppes_addresses/pl_secondary_practice_addresses_only.csv',
  '/Users/shashin/Documents/GitHub/INT Biohackathon 2026/data/nppes_addresses/endpoint_affiliation_addresses_only.csv']}

### Optional: normalize US ZIP+4 for display

US postal codes often appear as **9 digits** (e.g. `283044552`). This does not change the source data; it adds helper columns for readability.

In [8]:
import re


def format_us_postal(z: str | float | None) -> str | None:
    if z is None or (isinstance(z, float) and pd.isna(z)):
        return None
    s = str(z).strip()
    if not s or s.upper() == "NAN":
        return None
    digits = re.sub(r"\D", "", s)
    if len(digits) == 9 and digits.isdigit():
        return f"{digits[:5]}-{digits[5:]}"
    return s


pc_m = "Provider Business Mailing Address Postal Code"
pc_p = "Provider Business Practice Location Address Postal Code"
if pc_m in df_npi_preview.columns:
    df_npi_preview = df_npi_preview.assign(
        mailing_postal_formatted=df_npi_preview[pc_m].map(format_us_postal),
        practice_postal_formatted=df_npi_preview[pc_p].map(format_us_postal),
    )
df_npi_preview[["NPI", pc_m, "mailing_postal_formatted", pc_p, "practice_postal_formatted"]]

,NPI,Provider Business Mailing Address Postal Code,mailing_postal_formatted,Provider Business Practice Location Address Postal Code,practice_postal_formatted
0,1679576722,688482168,68848-2168,688472944,68847-2944
1,1588667638,322044736,32204-4736,322044736,32204-4736
2,1497758544,283044552,28304-4552,283044552,28304-4552
3,1306849450,NaN,NaN,NaN,NaN
4,1215930367,770901243,77090-1243,770901243,77090-1243
